# PySpark â€” Prerequisites & Environment Setup
### Run this notebook ONCE before starting any PySpark Day notebook.

---

This notebook:
1. Checks Java (required for Spark) and guides installation
2. Installs `pyspark` and supporting packages
3. Creates a test SparkSession
4. Runs a smoke test to confirm everything works
5. Explains how Day notebooks are structured

> PySpark requires **Java 8, 11, or 17**.  
> It does NOT require a Hadoop or Spark cluster â€” local mode is used throughout this sprint.

---
## SECTION A â€” Install Java (Required for Spark)

Spark is written in Scala and runs on the JVM. Python communicates with it via PySpark.  
Java must be installed before PySpark can work.

**Check if Java is already installed:**
```bash
java -version
```
If you see `openjdk version "11.x.x"` or `"17.x.x"` â€” skip to Section B.

---

### A1. Windows â€” Install Java

**Option 1 â€” Adoptium OpenJDK (recommended)**  
Download from: https://adoptium.net/  
Select: **Java 11 LTS** â†’ Windows â†’ x64 â†’ .msi installer  
Run the installer â†’ accept defaults â†’ check **"Set JAVA_HOME"** if offered.

**Set JAVA_HOME manually if not set by installer:**  
Open Start â†’ "Edit the system environment variables" â†’ Environment Variables  
Under **System variables** â†’ New:
```
Variable name:  JAVA_HOME
Variable value: C:\Program Files\Eclipse Adoptium\jdk-11.0.xx-hotspot
```
*(adjust path to match your actual install directory)*

Also add to `Path`: `%JAVA_HOME%\bin`

**Verify in a new PowerShell window:**
```powershell
java -version
echo $env:JAVA_HOME
```

---

### A2. macOS â€” Install Java

```bash
# Using Homebrew
brew install openjdk@11

# Add to PATH (add to ~/.zshrc or ~/.bash_profile)
echo 'export JAVA_HOME=$(/usr/libexec/java_home -v 11)' >> ~/.zshrc
echo 'export PATH="$JAVA_HOME/bin:$PATH"' >> ~/.zshrc
source ~/.zshrc

# Verify
java -version
echo $JAVA_HOME
```

---

### A3. Ubuntu / Debian â€” Install Java

```bash
# Install OpenJDK 11
sudo apt update
sudo apt install -y openjdk-11-jdk

# Verify
java -version

# Set JAVA_HOME (add to ~/.bashrc)
echo 'export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64' >> ~/.bashrc
echo 'export PATH="$JAVA_HOME/bin:$PATH"' >> ~/.bashrc
source ~/.bashrc

echo $JAVA_HOME
```

---

### A4. Verify Java from This Notebook

In [1]:
import os, sys

# Must be set BEFORE any pyspark import â€” Spark reads these at import time
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

print("JAVA_HOME        =", os.environ['JAVA_HOME'])
print("PYSPARK_PYTHON   =", os.environ['PYSPARK_PYTHON'])
print("Current Python   =", sys.executable)

JAVA_HOME        = C:/Program Files/DBeaver/jre
PYSPARK_PYTHON   = C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe
Current Python   = C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe


---
## SECTION B â€” Install PySpark and Python Packages

| Package | Purpose |
|---------|--------|
| `pyspark` | Apache Spark Python API |
| `pandas` | DataFrame display / interop |
| `pyarrow` | Efficient Spark â†” Pandas conversion |

After this cell completes, **Kernel â†’ Restart Kernel**, then continue from Section C.

In [2]:
import subprocess, sys

packages = [
    "pyspark",
    "pandas",
    "pyarrow",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + packages,
    capture_output=True, text=True
)

output = result.stdout + result.stderr
print(output[-3000:] if len(output) > 3000 else output)
print("\n--- RESTART KERNEL NOW: Kernel â†’ Restart Kernel, then continue from Section C ---")


[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


--- RESTART KERNEL NOW: Kernel â†’ Restart Kernel, then continue from Section C ---


---
## SECTION C â€” Start a Test SparkSession

> Run this AFTER restarting the kernel.

Every Day notebook starts with exactly this block â€” no additional config needed for local mode.

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Prerequisites_Test") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version  : {spark.version}")
print(f"Python version : {spark.sparkContext.pythonVer}")
print(f"Master         : {spark.sparkContext.master}")
print(f"App name       : {spark.sparkContext.appName}")

Spark version  : 3.5.6
Python version : 3.11
Master         : local[*]
App name       : Prerequisites_Test


**Common errors at this step:**

| Error | Cause | Fix |
|-------|-------|-----|
| `JAVA_HOME is not set` | Java not installed or JAVA_HOME missing | Install Java (Section A) and set JAVA_HOME |
| `No module named 'pyspark'` | Package not installed or kernel not restarted | Re-run Section B â†’ Restart Kernel â†’ re-run from Section C |
| `Port 4040 already in use` | Another Spark session is running | `spark.stop()` the previous one or restart kernel |
| Slow first start | Spark initializes JVM and downloads metadata | Normal â€” takes 20â€“60s on first run |

---
## SECTION D â€” Smoke Test

In [4]:
from pyspark.sql.functions import col, avg, count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

# Create a small test DataFrame
data = [
    (1, "Alice",  "Engineering",  95000.0),
    (2, "Bob",    "Engineering",  65000.0),
    (3, "Carol",  "Finance",      85000.0),
    (4, "Dave",   "Finance",      60000.0),
    (5, "Eve",    "HR",           75000.0),
]

schema = StructType([
    StructField("id",     IntegerType(), True),
    StructField("name",   StringType(),  True),
    StructField("dept",   StringType(),  True),
    StructField("salary", FloatType(),   True),
])

df = spark.createDataFrame(data, schema=schema)
print("DataFrame created:")
df.show()

DataFrame created:
+---+-----+-----------+-------+
| id| name|       dept| salary|
+---+-----+-----------+-------+
|  1|Alice|Engineering|95000.0|
|  2|  Bob|Engineering|65000.0|
|  3|Carol|    Finance|85000.0|
|  4| Dave|    Finance|60000.0|
|  5|  Eve|         HR|75000.0|
+---+-----+-----------+-------+



In [5]:
# Test filter, groupBy, agg
print("Engineering employees with salary > 70000:")
df.filter(
    (col("dept") == "Engineering") & (col("salary") > 70000)
).show()

print("Average salary by department:")
df.groupBy("dept").agg(
    avg("salary").alias("avg_salary"),
    count("*").alias("headcount")
).orderBy("avg_salary").show()

Engineering employees with salary > 70000:
+---+-----+-----------+-------+
| id| name|       dept| salary|
+---+-----+-----------+-------+
|  1|Alice|Engineering|95000.0|
+---+-----+-----------+-------+

Average salary by department:
+-----------+----------+---------+
|       dept|avg_salary|headcount|
+-----------+----------+---------+
|    Finance|   72500.0|        2|
|         HR|   75000.0|        1|
|Engineering|   80000.0|        2|
+-----------+----------+---------+



In [7]:
# Test SparkSQL
df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT dept, ROUND(AVG(salary), 0) AS avg_salary
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
""").show()

print("All smoke tests passed. PySpark is working correctly.")

+-----------+----------+
|       dept|avg_salary|
+-----------+----------+
|Engineering|   80000.0|
|         HR|   75000.0|
|    Finance|   72500.0|
+-----------+----------+

All smoke tests passed. PySpark is working correctly.


---
## SECTION E â€” Understanding Local Mode

All Day notebooks run Spark in **local mode** â€” your laptop acts as both driver and executor.

```python
.master("local[*]")   # use all available CPU cores
.master("local[2]")   # use exactly 2 cores
.master("local")      # use 1 core
```

**Key settings used in every Day notebook:**

| Config | Value | Why |
|--------|-------|-----|
| `spark.sql.shuffle.partitions` | `4` | Default is 200 â€” reduces overhead for small local data |
| `setLogLevel("ERROR")` | ERROR | Hides INFO/WARN noise from output |

**Spark UI** â€” while a SparkSession is running, view the web UI at:  
`http://localhost:4040` â€” shows jobs, stages, DAG, and SQL query plans.

---
## SECTION F â€” Key Import Pattern for Day Notebooks

Every Day notebook uses the same header â€” this is what you'll see at the top of each one:

In [9]:
# Standard header â€” every Day notebook starts with exactly this
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, lit,
    avg, count, sum as spark_sum, min as spark_min, max as spark_max,
    round as spark_round,
    upper, lower, trim, regexp_replace, substring, concat, length,
    to_date, year, month, datediff, date_trunc, date_format,
    row_number, rank, dense_rank, lag, lead, ntile,
    when, isnull, coalesce
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, FloatType, DoubleType, DateType, BooleanType
)

print("All PySpark imports â€” OK")

All PySpark imports â€” OK


---
## SECTION G â€” Windows-Specific: Hadoop Winutils (if needed)

On **Windows**, PySpark may print a warning about missing `winutils.exe`.  
This usually doesn't stop the code from working, but if you see errors about `hadoop.tmp.dir`, fix it:

**Option 1 â€” Suppress the warning (easiest)**  
The `setLogLevel("ERROR")` call in every Day notebook already suppresses this.

**Option 2 â€” Install winutils (fully fix it)**

```powershell
# 1. Create a folder
mkdir C:\hadoop\bin

# 2. Download winutils.exe matching your Hadoop version
#    PySpark 3.x ships with Hadoop 3.x
#    Download from: https://github.com/cdarlint/winutils
#    Get: hadoop-3.3.5/bin/winutils.exe
#    Place it at: C:\hadoop\bin\winutils.exe

# 3. Set environment variables (System variables)
# HADOOP_HOME = C:\hadoop
# Add to Path: %HADOOP_HOME%\bin
```

After setting these, restart your terminal and re-open Jupyter.

---
## All Done â€” Day Notebook Structure

```
pyspark/
â”œâ”€â”€ 00_prerequisites.ipynb                 â† this file
â”œâ”€â”€ day01_dataframe_filtering/
â”‚   â”œâ”€â”€ notes.md                           â† read first
â”‚   â””â”€â”€ notebook.ipynb                     â† run second
â”œâ”€â”€ day02_.../
â”‚   â””â”€â”€ ...
```

**Workflow each day:**
1. Read `notes.md` â€” understand the PySpark API and how it maps to SQL
2. Open `notebook.ipynb` â€” run each cell, study the output
3. Try solving the Day problems yourself before looking at solution cells

Each Day notebook creates its own SparkSession in the first cell â€” no shared state between notebooks.

In [10]:
spark.stop()
print("Test SparkSession stopped. Setup complete.")

Test SparkSession stopped. Setup complete.
